[<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/9_grasp_learning_cmaes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HSR Grasp Parameter Learning with CMA-ES / CMA-ESによる把持パラメータ学習

**目的 / Objective:**

- **CMA-ES** (Covariance Matrix Adaptation Evolution Strategy) を用いて、IK把持パイプラインの把持パラメータを最適化する方法を学ぶ / Learn to optimize grasp parameters of the IK pick pipeline using **CMA-ES**.
- 7種類のYCB物体に対して、物体ごとの最適な把持高さ・グリッパ力・保持時間を進化戦略で探索する / Search for per-object optimal grasp height, gripper effort, and hold time across 7 YCB objects using an evolution strategy.
- EvoTorchのGPU加速CMA-ESとGenesis並列シミュレーションを組み合わせた高速パラメータ最適化を体験する / Experience fast parameter optimization combining EvoTorch's GPU-accelerated CMA-ES with Genesis parallel simulation.

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better). Genesis's parallel simulation lets Colab GPUs handle popsize=256 as well, not just local GPUs — adjust `GENERATIONS` to fit your session length.

## 概要 / Overview

前のチュートリアルでは、IK把持パイプライン (アプローチ → 降下 → 把持 → 持ち上げ) を固定パラメータで実行しました。このノートブックでは、**把持パラメータを学習可能**にし、CMA-ESで最適化します。

In the previous tutorial, the IK grasp pipeline ran with fixed parameters. This notebook makes the **grasp parameters learnable** and optimizes them with CMA-ES.

### 最適化するパラメータ / Parameters to optimize

| Parameter | Range | Description |
|-----------|-------|-------------|
| `pre_grasp_height` | 0.05–0.30 m | Hover height above object before descending / 降下前のホバー高さ |
| `grasp_offset_z` | -0.02–0.08 m | Final grasp height relative to object center / 最終把持高さ |
| `gripper_effort` | 1.0–8.0 N | Force applied when closing gripper / グリッパ閉鎖時の力 |
| `grasp_hold_steps` | 100–500 steps | Steps to hold while gripper closes / 把持保持ステップ数 |

各YCB物体 (foam_brick, apple, banana, orange, tennis_ball, baseball, rubiks_cube) に対して独立した4パラメータを最適化します (合計28次元)。

We optimize 4 independent parameters per YCB object (28 dimensions total).

## 1. Setup / セットアップ

依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
Install dependencies, clone the repo, and configure GPU rendering.

In [ ]:
import importlib, urllib.request

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering.
setup_colab()

# Install EvoTorch for CMA-ES
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'evotorch', '-q'], check=True)
print('EvoTorch installed.')

In [ ]:
import sys, pathlib

# Add repo to path (same as other tutorials)
REPO_DIR = pathlib.Path('/content/hsr-genesis')
SRC_DIR = str(REPO_DIR / 'src')
EXAMPLES_DIR = str(REPO_DIR / 'examples' / 'rl')
for p in [SRC_DIR, EXAMPLES_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import genesis as gs
if not getattr(gs, '_initialized', False):
    gs.init(backend=gs.gpu)
else:
    print('Genesis already initialized.')

import torch
import numpy as np
print(f'Genesis device: {gs.device}')

## 2. Grasp Parameters / 把持パラメータ

把持パラメータの定義と範囲を確認します。28次元ベクトル = 4パラメータ × 7物体。
Inspect the grasp parameter definitions and bounds. 28D vector = 4 params × 7 objects.

In [ ]:
from grasp_params import (
    PARAM_NAMES, OBJECT_NAMES, PARAM_BOUNDS,
    SOLUTION_LENGTH, N_PARAMS, N_OBJECTS,
    PARAM_DEFAULTS, default_params, denormalize, params_to_dict,
)

print(f'Parameters: {PARAM_NAMES}')
print(f'Objects:    {OBJECT_NAMES}')
print(f'Solution length: {SOLUTION_LENGTH} (={N_PARAMS} x {N_OBJECTS})')
print()
print('Parameter bounds:')
for i, name in enumerate(PARAM_NAMES):
    lo, hi = PARAM_BOUNDS[i]
    print(f'  {name:25s} [{lo:.2f}, {hi:.2f}]')
print()
print(f'Default params (all objects): {PARAM_DEFAULTS.tolist()}')

## 3. Baseline Evaluation / ベースライン評価

デフォルトパラメータで全物体の把持成功率を測定します。これが最適化の開始点となります。
Measure pick success rates with default parameters across all objects. This is the optimization starting point.

### カーネルウォームアップ / Kernel Warmup

Colabの初回実行時、Quadrants（GenesisのGPUバックエンド）がNVRTCでGPUカーネルをJITコンパイルするため、最初の`scene.build()`に数分かかります。小さなシーンをビルドしてカーネルをコンパイルした後、`gs.destroy()` → `gs.init()`でキャッシュをディスクにフラッシュします。これにより以降のシーンビルドが高速化されます。

On first Colab run, Quadrants JIT-compiles GPU kernels via NVRTC, making the first `scene.build()` take several minutes. The warmup below builds a tiny scene to trigger compilation, then calls `gs.destroy()` (which flushes the PTX cache to disk via `qd.reset()`) and `gs.init()` to reload it. Subsequent builds reuse the cached PTX.

In [ ]:
# Warmup: build a tiny scene and run a mini pick pipeline to pre-compile
# ALL quadrants GPU kernels (NVRTC), then destroy + re-init Genesis to
# flush the PTX cache to disk.
#
# On Colab, the quadrants PTX cache starts empty. The first call to each
# unique GPU kernel triggers NVRTC JIT compilation, which takes seconds
# per kernel with no log output. scene.step() alone only compiles physics
# step kernels — IK, trajectory control, gripper force, and state readback
# each have their own kernels that are only compiled when first called.
# We must exercise the full pick pipeline to compile everything.
import time as _time
from ycb_pick_ik_parallel import HSRPickEnv

print('Warming up quadrants GPU kernel cache (one-time cost on Colab)...')
print('  Building 2-env scene + running mini pipeline to compile all kernels...')
_t0 = _time.time()

# 1. Build a tiny scene — triggers scene.build() kernel compilation.
_warmup_env = HSRPickEnv(
    n_envs=2,
    object_name=OBJECT_NAMES[0],
    show_viewer=False,
    seed=0,
    disable_visualizer=True,
)

# 2. Run the full pick pipeline with a small settle — this exercises
#    IK, trajectory control, gripper force, state readback, and all
#    simulation step kernels, triggering NVRTC compilation for each.
_warmup_env.grasp_params = None  # use defaults
_warmup_env.run_pick_pipeline(settle_steps=2)
del _warmup_env

# 3. Destroy Genesis — flushes compiled PTX kernels to disk via qd.reset().
gs.destroy()

# 4. Re-init — loads the persisted PTX cache so all future builds are fast.
gs.init(backend=gs.gpu)

print(f'Warmup done ({_time.time() - _t0:.1f}s). '
      'All kernels compiled & cached. Subsequent runs will be fast.')

In [ ]:
from ycb_pick_ik_parallel import HSRMultiObjectPickEnv
import contextlib, io

N_ENVS = 256  # split ~evenly across all 7 objects in one fused scene
SETTLE_STEPS = 30

default_matrix = default_params().to(gs.device)  # (7, 4)

# One scene, all 7 objects spawned once; each env is round-robin assigned
# an "active" object (the other 6 are parked away from the robot). This
# evaluates all objects in a single batched pipeline run instead of
# rebuilding the scene and re-running the pipeline once per object.
with contextlib.redirect_stdout(io.StringIO()):
    baseline_env = HSRMultiObjectPickEnv(
        n_envs=N_ENVS,
        object_names=OBJECT_NAMES,
        show_viewer=False,
        seed=42,
        disable_visualizer=True,
    )
    baseline_env.grasp_params = default_matrix[baseline_env.env_object_idx]
    baseline_env.run_pick_pipeline(settle_steps=SETTLE_STEPS)

baseline_summary = baseline_env.get_per_object_summary()
baseline_rates = {name: baseline_summary[name]['success_rate'] for name in OBJECT_NAMES}
for obj_name in OBJECT_NAMES:
    s = baseline_summary[obj_name]
    print(f'  {obj_name:25s} {s["success_rate"]:.2%}  (n={s["n_envs"]})', flush=True)

baseline_mean = np.mean(list(baseline_rates.values()))
print(f'\nBaseline mean success: {baseline_mean:.2%}')

## 4. CMA-ES Optimizer / CMA-ES最適化器

EvoTorchのCMA-ESを使用します。各世代で256個の候補パラメータをサンプリングし、7物体すべてで評価して適応度を計算します。

We use EvoTorch's CMA-ES. Each generation samples 256 candidate parameter vectors, evaluates them on all 7 objects, and computes fitness.

### アルゴリズムの仕組み / How it works

1. **サンプリング / Sampling**: CMA-ESが多変量正規分布から候補を生成 / CMA-ES samples candidates from a multivariate Gaussian
2. **評価 / Evaluation**: 各候補を7物体でIK把持パイプライン実行 / Run IK pick pipeline for each candidate on 7 objects
3. **適応度 / Fitness**: 7物体の平均成功率 / Mean success rate across 7 objects
4. **更新 / Update**: 適応度に基づいて分布の平均・共分散・ステップサイズを更新 / Update distribution mean, covariance, step size based on fitness

![CMA-ES concept](https://raw.githubusercontent.com/CMA-ES/pycma/master/doc/cmaes-flow.png)

In [ ]:
from evotorch import Problem
from evotorch.algorithms import CMAES
from ycb_pick_ik_parallel import HSRPickEnv

class GraspProblem(Problem):
    """EvoTorch Problem: evaluate grasp params via IK pick simulation."""

    def __init__(self, *, popsize, settle_steps, seed=0):
        super().__init__(
            objective_sense='max',
            solution_length=SOLUTION_LENGTH,
            initial_bounds=(0.0, 1.0),  # normalized search space
            device='cpu',
        )
        self.popsize = popsize
        self.settle_steps = settle_steps
        self.seed = seed
        self._envs = {}

    def _get_env(self, object_name):
        if object_name not in self._envs:
            self._envs[object_name] = HSRPickEnv(
                n_envs=self.popsize,
                object_name=object_name,
                show_viewer=False,
                seed=self.seed,
                disable_visualizer=True,
            )
        return self._envs[object_name]

    def _evaluate_batch(self, solutions):
        n = solutions.values.shape[0]
        raw = solutions.values.clone()  # (n, 28) in [0,1]

        # Denormalize [0,1] -> actual param ranges
        lo = PARAM_BOUNDS[:, 0].repeat(N_OBJECTS).to(raw.device)
        hi = PARAM_BOUNDS[:, 1].repeat(N_OBJECTS).to(raw.device)
        scaled = raw.clamp(0.0, 1.0) * (hi - lo) + lo
        clipped = scaled.reshape(n, N_OBJECTS, N_PARAMS)
        clipped[..., 3] = torch.round(clipped[..., 3])  # round hold steps to int

        success_per_obj = torch.zeros(n, N_OBJECTS, dtype=torch.float32)

        for obj_idx, obj_name in enumerate(OBJECT_NAMES):
            env = self._get_env(obj_name)
            env.grasp_params = clipped[:, obj_idx, :].to(gs.device, dtype=gs.tc_float)
            result = env.run_pick_pipeline(settle_steps=self.settle_steps)
            success_per_obj[:, obj_idx] = result['success_per_env']

        fitness = success_per_obj.mean(dim=1)
        solutions.set_evals(fitness)

        best_idx = int(fitness.argmax())
        per_obj = [f'{float(success_per_obj[:, i].mean()):.2f}' for i in range(N_OBJECTS)]
        print(f'  [eval] best={float(fitness[best_idx]):.3f} '
              f'mean={float(fitness.mean()):.3f} per_obj={per_obj}')

print('GraspProblem defined.')

### CMA-ES vs 遺伝的アルゴリズム / CMA-ES vs Genetic Algorithm

なぜCMA-ESがこの問題に適しているのか、遺伝的アルゴリズム (GA) と比較して理解しましょう。

Let's understand why CMA-ES is well-suited for this problem by contrasting it with a Genetic Algorithm (GA).

| Aspect / 側面 | Genetic Algorithm (GA) | CMA-ES |
|---|---|---|
| **表現 / Representation** | 個体を染色体（離散/連続）として符号化 / Encode individuals as chromosomes (discrete/continuous) | 連続ベクトル（多変量ガウス分布） / Continuous vector (multivariate Gaussian) |
| **仕組み / Mechanism** | 選択→交差→突然変異を個体に適用 / Selection → Crossover → Mutation on individuals | ガウス分布 N(m, C, σ²) からサンプリング→適応度で平均・共分散・ステップサイズを更新 / Sample from Gaussian, update mean, covariance, step size |
| **適応 / Adaptation** | 突然変異率は固定（またはスケジュール） / Mutation rate is fixed (or scheduled) | ステップサイズ σ と共分散行列 C が**自動適応** / Step size σ and covariance matrix C **automatically adapt** |
| **活用情報 / Info used** | 個体の優劣（適応度ランキング）のみ / Only individual superiority (fitness ranking) | ランキング**＋分布構造**（変数間相関） / Ranking **+ distribution structure** (inter-variable correlations) |
| **勾配 / Gradient** | 不要 / Not required | 不要（ブラックボックス最適化）/ Not required (black-box optimization) |
| **適した空間 / Best search space** | 離散/組合せ最適化（ビット列、順列） / Discrete/combinatorial (bit strings, permutations) | 連続値（実数ベクトル） / Continuous (real-valued vector) |
| **集団 / Population** | 固定サイズ、トーナメント/ランキング選択 / Fixed size, tournament/ranking selection | 適応的サンプリング、集団サイズ可変 / Adaptive sampling, changeable popsize |
| **収束 / Convergence** | 突然変異率固定だと最適解付近で減速 / Slows near optimum when mutation rate is fixed | 最適解への幾何収束（理論保証）/ Geometric convergence to optimum (proven) |

### なぜこの問題にCMA-ESか / Why CMA-ES for this problem

- **28次元連続サーチ空間**: 4パラメータ × 7物体 = 実数ベクトル → GAの離散交差より連続ガウスサンプリングが自然 / 28D continuous space: real-valued vector → continuous Gaussian sampling more natural than GA's discrete crossover
- **変数間相関のキャプチャ**: 把持高さが低い↔グリッパ力を高く必要、などの相関を共分散行列が学習 / Covariance captures correlations (e.g. low grasp height ↔ higher effort needed)
- **疎な報酬**: 成功/失敗の二値報酬 → 勾配ベース法が困難だが、ランキングベースのCMA-ESは機能 / Sparse binary reward → gradient methods struggle, but ranking-based CMA-ES works
- **軸非整列の相関**: GAの交差は軸に沿った組み合わせしか生成しないが、CMA-ESの共分散行列は任意方向の相関を表現可能 / GA crossover only combines along axes; CMA-ES covariance can represent correlations in any direction

### 1次元デモ: GA vs CMA-ES / 1D Demo: GA vs CMA-ES

1次元の多峰性地形で両アルゴリズムの挙動を可視化します。GAは集団が良い領域に集中する様子、CMA-ESはガウス分布が最適解に向かって収縮する様子が観察できます。

Visualize both algorithms on a 1D multimodal landscape. Observe GA's population clustering vs CMA-ES's Gaussian distribution shrinking toward the optimum.

In [ ]:
# 1D landscape: multimodal function
def landscape(x):
    return np.sin(3 * x) * np.exp(-0.1 * x**2)

x_grid = np.linspace(-5, 5, 300)
y_grid = landscape(x_grid)

# --- GA simulation ---
rng_ga = np.random.default_rng(0)
pop_ga = rng_ga.uniform(-5, 5, 30)
ga_history = [pop_ga.copy()]
for _ in range(20):
    fitness = landscape(pop_ga)
    # Tournament: keep top 50%
    idx = np.argsort(fitness)[::-1][:15]
    parents = pop_ga[idx]
    # Crossover (pick from parents) + mutation
    children = rng_ga.choice(parents, 15) + rng_ga.normal(0, 0.5, 15)
    pop_ga = np.concatenate([parents, children])
    ga_history.append(pop_ga.copy())

# --- CMA-ES simulation (simplified 1D) ---
rng_cma = np.random.default_rng(0)
mean = 0.0
sigma = 2.0
cma_means = []
cma_samples_history = []
for _ in range(20):
    samples = rng_cma.normal(mean, sigma, 30)
    fitness = landscape(samples)
    # Rank-based weighted mean update
    idx = np.argsort(fitness)[::-1]
    mu = 15
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= weights.sum()
    mean = np.sum(weights * samples[idx[:mu]])
    # Simple step-size adaptation
    success_rate = np.mean(fitness > landscape(np.full(30, mean)))
    sigma *= np.exp(0.1 * (success_rate - 0.2))
    sigma = max(sigma, 0.01)
    cma_means.append(mean)
    cma_samples_history.append(samples.copy())
cma_means.append(mean)
cma_samples_history.append(rng_cma.normal(mean, sigma, 30))

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# GA
ax = axes[0]
ax.plot(x_grid, y_grid, 'k-', alpha=0.3, linewidth=2)
for i, pop in enumerate(ga_history):
    color = plt.cm.viridis(i / len(ga_history))
    ax.scatter(pop, landscape(pop), color=color, alpha=0.5, s=20)
ax.set_title('Genetic Algorithm: population scatter per generation')
ax.set_xlabel('x')
ax.set_ylabel('fitness')
ax.set_xlim(-5, 5)

# CMA-ES
ax = axes[1]
ax.plot(x_grid, y_grid, 'k-', alpha=0.3, linewidth=2)
for i, (mean_v, samples) in enumerate(zip(cma_means, cma_samples_history)):
    color = plt.cm.viridis(i / len(cma_samples_history))
    ax.scatter(samples, landscape(samples), color=color, alpha=0.5, s=20)
    ax.axvline(mean_v, color=color, alpha=0.3, linestyle='--', linewidth=1)
ax.set_title('CMA-ES: Gaussian samples shrink around optimum')
ax.set_xlabel('x')
ax.set_ylabel('fitness')
ax.set_xlim(-5, 5)

plt.tight_layout()
plt.show()

print('観察ポイント / Observations:')
print('  GA: 各世代で親選択+突然変異 → 良い領域に集団が集中 / Population clusters around good regions via parent selection + mutation')
print('  CMA-ES: ガウス分布のσが縮小 → 最適解付近にサンプリングが集中 / Gaussian σ shrinks → samples concentrate near optimum')

## 5. Training / 学習

CMA-ESで把持パラメータを最適化します。Genesisの並列シミュレーションのおかげで、Colab GPU上でも popsize=256 を使用できます。

Optimize grasp parameters with CMA-ES. Thanks to Genesis's parallel simulation, popsize=256 works on Colab GPUs too, not just local ones.

> **Tip:** `GENERATIONS` を増やす（例: 50）とより良い結果が得られますが、Colabのセッション時間に注意してください / Increase `GENERATIONS` (e.g. to 50) for better results, but keep Colab's session time limits in mind.

In [ ]:
import time, json

POPSIZE = 256      # Genesis parallel sim handles this on Colab GPUs too
GENERATIONS = 5    # increase (e.g. to 50) for better results, session time permitting
SETTLE = 30
OUTPUT_DIR = pathlib.Path('/content/grasp_cmaes_results')
OUTPUT_DIR.mkdir(exist_ok=True)

problem = GraspProblem(popsize=POPSIZE, settle_steps=SETTLE, seed=0)

# Initialize CMA-ES at normalized default params
lo_all = PARAM_BOUNDS[:, 0].repeat(N_OBJECTS)
hi_all = PARAM_BOUNDS[:, 1].repeat(N_OBJECTS)
defaults_norm = (PARAM_DEFAULTS.repeat(N_OBJECTS) - lo_all) / (hi_all - lo_all)

cmaes = CMAES(
    problem=problem,
    stdev_init=0.2,
    popsize=POPSIZE,
    center_init=defaults_norm,
)

best_fitness = -1.0
best_params = None
history = []

for gen in range(GENERATIONS):
    t0 = time.time()
    cmaes.step()
    pop = cmaes.population
    evals = pop.evals
    best_f = float(evals.max())
    mean_f = float(evals.mean())
    dt = time.time() - t0

    if best_f > best_fitness:
        best_fitness = best_f
        best_idx = int(evals.argmax())
        best_raw = pop.values[best_idx].clone().cpu()
        scaled = best_raw.clamp(0, 1) * (hi_all - lo_all) + lo_all
        best_matrix = scaled.reshape(N_OBJECTS, N_PARAMS)
        best_matrix[:, 3] = torch.round(best_matrix[:, 3])
        best_params = params_to_dict(best_matrix)

    history.append({'gen': gen, 'best': best_f, 'mean': mean_f})
    print(f'[gen {gen:>3d}] best={best_f:.3f} mean={mean_f:.3f} '
          f'({dt:.1f}s) overall_best={best_fitness:.3f}')

    # Checkpoint
    with open(OUTPUT_DIR / 'grasp_cmaes_best.json', 'w') as f:
        json.dump({'generation': gen, 'fitness': best_fitness, 'params': best_params}, f, indent=2)

print(f'\nTraining complete. Best fitness: {best_fitness:.3f}')
print(f'Baseline was: {baseline_mean:.3f}')
print(f'Improvement:  {best_fitness - baseline_mean:+.3f}')

## 6. Training Curve / 学習曲線

世代ごとの適応度推移を可視化します。
Visualize fitness progression across generations.

In [ ]:
import matplotlib.pyplot as plt

gens = [h['gen'] for h in history]
bests = [h['best'] for h in history]
means = [h['mean'] for h in history]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(gens, bests, 'o-', label='Best fitness', color='tab:blue')
ax.plot(gens, means, 's--', label='Mean fitness', color='tab:orange', alpha=0.7)
ax.axhline(y=baseline_mean, color='tab:red', linestyle=':', label=f'Baseline ({baseline_mean:.2f})')
ax.set_xlabel('Generation')
ax.set_ylabel('Mean success rate (7 objects)')
ax.set_title('CMA-ES Grasp Parameter Optimization')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Learned Parameters / 学習済みパラメータ

最適化されたパラメータをデフォルトと比較します。
Compare optimized parameters against defaults.

In [ ]:
print('Learned grasp parameters:')
print(f'{"Object":25s} {"pre_grasp":>10s} {"grasp_z":>10s} {"effort":>10s} {"hold":>8s}')
print('-' * 65)
for obj_name in OBJECT_NAMES:
    p = best_params[obj_name]
    print(f'{obj_name:25s} {p["pre_grasp_height"]:10.4f} {p["grasp_offset_z"]:10.4f} '
          f'{p["gripper_effort"]:10.2f} {p["grasp_hold_steps"]:8d}')

print(f'\nDefaults: {PARAM_DEFAULTS.tolist()}')
print(f'\nObject-specific insights:')
for obj_name in OBJECT_NAMES:
    p = best_params[obj_name]
    effort_diff = p['gripper_effort'] - float(PARAM_DEFAULTS[2])
    height_diff = p['pre_grasp_height'] - float(PARAM_DEFAULTS[0])
    notes = []
    if effort_diff > 1.0:
        notes.append(f'+{effort_diff:.1f}N effort')
    elif effort_diff < -1.0:
        notes.append(f'{effort_diff:.1f}N effort')
    if height_diff > 0.05:
        notes.append(f'+{height_diff:.2f}m height')
    elif height_diff < -0.05:
        notes.append(f'{height_diff:.2f}m height')
    if notes:
        print(f'  {obj_name:25s} {" ".join(notes)}')

## 8. Final Evaluation / 最終評価

学習済みパラメータで全物体を評価し、ベースラインと比較します。
Evaluate all objects with learned parameters and compare against baseline.

In [ ]:
from grasp_params import params_from_dict
from ycb_pick_ik_parallel import HSRMultiObjectPickEnv

param_matrix = params_from_dict(best_params).to(gs.device)

with contextlib.redirect_stdout(io.StringIO()):
    final_env = HSRMultiObjectPickEnv(
        n_envs=N_ENVS,
        object_names=OBJECT_NAMES,
        show_viewer=False,
        seed=42,
        disable_visualizer=True,
    )
    final_env.grasp_params = param_matrix[final_env.env_object_idx]
    final_env.run_pick_pipeline(settle_steps=SETTLE_STEPS)

learned_summary = final_env.get_per_object_summary()
learned_rates = {name: learned_summary[name]['success_rate'] for name in OBJECT_NAMES}

print(f'{"Object":25s} {"Baseline":>10s} {"Learned":>10s} {"Delta":>8s}')
print('-' * 55)

for obj_name in OBJECT_NAMES:
    delta = learned_rates[obj_name] - baseline_rates[obj_name]
    print(f'{obj_name:25s} {baseline_rates[obj_name]:10.2%} {learned_rates[obj_name]:10.2%} {delta:+8.2%}', flush=True)

learned_mean = np.mean(list(learned_rates.values()))
print(f'\n{"Mean":25s} {baseline_mean:10.2%} {learned_mean:10.2%} {learned_mean - baseline_mean:+8.2%}')

## 9. Per-Object Comparison Bar Chart / 物体別比較

ベースラインと学習済みの成功率を棒グラフで比較します。
Compare baseline vs learned success rates with a bar chart.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

x = np.arange(len(OBJECT_NAMES))
width = 0.35

baseline_vals = [baseline_rates[o] for o in OBJECT_NAMES]
learned_vals = [learned_rates[o] for o in OBJECT_NAMES]

short_names = [o.replace('ycb_', '') for o in OBJECT_NAMES]

bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline (defaults)', color='tab:red', alpha=0.7)
bars2 = ax.bar(x + width/2, learned_vals, width, label='Learned (CMA-ES)', color='tab:blue', alpha=0.7)

ax.set_ylabel('Success rate')
ax.set_title('Grasp Success: Baseline vs CMA-ES Learned Parameters')
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.0%}',
            ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.0%}',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 10. Save & Load / 保存と読み込み

学習済みパラメータを保存・読み込みする方法を示します。
Show how to save and load learned parameters.

In [ ]:
# Save is already done during training, but show the format:
checkpoint_path = OUTPUT_DIR / 'grasp_cmaes_best.json'
print(f'Checkpoint saved at: {checkpoint_path}')
print()

with open(checkpoint_path) as f:
    ckpt = json.load(f)
print(f'Generation: {ckpt["generation"]}')
print(f'Fitness:    {ckpt["fitness"]:.3f}')
print(f'Objects:    {list(ckpt["params"].keys())}')
print()
print('To load in your own script:')
print('''
  import json
  from grasp_params import params_from_dict
  with open("grasp_cmaes_best.json") as f:
      ckpt = json.load(f)
  param_matrix = params_from_dict(ckpt["params"])  # (7, 4) tensor
  # Use param_matrix[obj_idx] as grasp_params for HSRPickEnv
''')

## まとめ / Summary

CMA-ESによる把持パラメータ学習の流れを振り返ります / Review of CMA-ES grasp parameter learning:

| Step | Component | Description |
|------|-----------|-------------|
| Parameters | `grasp_params.py` | 4 params × 7 objects = 28D search space / 28次元探索空間 |
| Environment | `HSRPickEnv` | Parameterized IK pick pipeline / パラメータ化IK把持パイプライン |
| Optimizer | `GraspProblem` + `CMAES` | EvoTorch GPU CMA-ES / EvoTorch GPU CMA-ES |
| Training | `train_grasp_cmaes.py` | Training loop with logging & checkpointing / ログ・チェックポイント付き学習ループ |
| Evaluation | `eval_grasp_params.py` | Load checkpoint, eval on all objects / チェックポイント読み込み・全物体評価 |

### 重要なポイント / Key takeaways

- **CMA-ESはブラックボックス最適化**: 勾配不要、報酬が離散的（成功/失敗）でも機能する / CMA-ES is black-box optimization: no gradients needed, works with sparse binary rewards
- **並列シミュレーションが鍵**: 256環境の並列IK把持で1世代を数分で完了（Colab GPUでも可能） / Parallel simulation is key: 256-env parallel IK pick completes a generation in minutes (works on Colab GPUs too)
- **物体固有の戦略**: 同じパラメータではなく、物体ごとに異なる最適値を学習 / Learns object-specific strategies, not one-size-fits-all
- **限界**: テニスボールなど球形物体はトップダウン把持では困難 — 戦略の変更が必要 / Limitation: round objects (tennis ball) are hard with top-down grasp — need strategy change

### 次のステップ / Next steps

- **より大規模な訓練 / Larger training**: `--generations 50` （またはそれ以上）を実行してより良い結果を得る / Run `--generations 50` (or more) for better results
- **パラメータの追加 / More parameters**: アプローチ時間・降下時間も最適化対象に追加 / Add approach/descend durations to the search space
- **残差ポリシー / Residual policy**: IKの上にNNポリシーで補正を学習する次のステップへ / Next step: learn NN residual corrections on top of IK
- **把持戦略の選択 / Grasp strategy selection**: トップダウン・サイド・ピンチの戦略選択を学習 / Learn to select grasp strategies (top, side, pinch)